# ETL Data Mentah → Data Mart DuckDB → Power BI

Notebook ini melanjutkan `generate_data_mentah_procurement.ipynb`:

1. **Extract**: baca CSV mentah di folder `raw_data/`.
2. **Transform**: bersihkan data kotor, bentuk dimensi (SCD 1/2/3) dan tabel fakta.
3. **Load**: simpan star schema ke file **`procurement_datamart.duckdb`** (+ salinan CSV di `datamart_csv/`).
4. **Koneksi ke Power BI**: membuat paket `koneksi_powerbi/` (skrip DSN, Power Query, DAX) + panduan di akhir notebook.

Struktur tabel **mengikuti ERD star schema kelompok** (1 fakta + 5 dimensi, nama & tipe kolom sama):

| Tabel | Kolom sesuai ERD | Kolom tambahan (wajib untuk SCD) |
|---|---|---|
| `fact_purchase_order` | order_id, product_key, vendor_key, contract_key, po_date_key, delivery_date_key, po_profile_key, po_number, quantity_ordered, unit_price, gross_amount, tax_amount, net_amount | – |
| `dim_vendor` | vendor_key, vendor_id, vendor_name, vendor_category, city, region | effective_date, expiry_date, is_current (SCD 2) |
| `dim_product` | product_key, product_id, product_name, product_type, allocation_line, unit_of_measure | effective_date, expiry_date, is_current (SCD 2) |
| `dim_contract` | contract_key, contract_id, contract_type, price_agreement_type, max_quota_limit, sla_temperature_requirement, is_active | start_date, end_date, prev_end_date, prev_max_quota_limit (SCD 3) |
| `dim_po_profile` | po_profile_key, po_status, urgency_level, is_contract_locked | – |
| `dim_date` | date_key, full_date, day_of_month, month_number, month_name, quarter, year | – |

```
raw_data/*.csv ──► ETL (notebook ini) ──► procurement_datamart.duckdb ──► ODBC ──► Power BI Desktop
```

## 0. Import & pengaturan

In [1]:
# !pip install duckdb pandas numpy
import os
import numpy as np
import pandas as pd
import duckdb

RAW_DIR   = 'raw_data'                     # hasil notebook generator
DB_PATH   = 'procurement_datamart.duckdb'  # file database untuk Power BI
CSV_DIR   = 'datamart_csv'                 # cadangan: star schema dalam CSV
TGL_EKSTRAK = pd.Timestamp('2026-09-25')   # tanggal data diambil dari sistem sumber
TGL_AWAL_SCD = pd.Timestamp('2023-01-01')  # effective_date versi pertama dimensi
TGL_MAKS     = pd.Timestamp('9999-12-31')  # expiry_date versi yang masih berlaku
KEY_BELUM_DITERIMA = 99991231              # date key untuk barang yang belum diterima (sama dengan FIX.zip)

## 1. Extract: baca data mentah

In [2]:
baca = lambda nama, tgl=(): pd.read_csv(f'{RAW_DIR}/{nama}.csv', parse_dates=list(tgl))
src_vendor   = baca('src_vendor')
src_product  = baca('src_product')
src_contract = baca('src_contract', ['start_date', 'end_date', 'signed_date'])
src_po       = baca('src_purchase_order', ['po_date', 'promised_date'])
src_gr       = baca('src_goods_receipt', ['receive_date'])
src_log      = baca('src_change_log', ['changed_at'])

for nama, df in [('src_vendor', src_vendor), ('src_product', src_product), ('src_contract', src_contract),
                 ('src_purchase_order', src_po), ('src_goods_receipt', src_gr), ('src_change_log', src_log)]:
    print(f'{nama:20s}: {len(df):6,d} baris')

src_vendor          :     20 baris
src_product         :     18 baris
src_contract        :     76 baris
src_purchase_order  : 10,010 baris
src_goods_receipt   :  9,464 baris
src_change_log      :    108 baris


## 2. Transform (a): pembersihan data (data cleaning)
Aturan pembersihan dibuat **berdasarkan pola data**, bukan membaca `outlier_log.csv`.
`outlier_log.csv` hanya dipakai di akhir untuk mengecek apakah pembersihan sudah tepat.

| No | Masalah | Aturan |
|---|---|---|
| 1 | `vendor_id` berspasi / huruf kecil | `strip()` + `upper()` |
| 2 | Baris PO duplikat | hapus duplikat berdasarkan `po_number` + `line_no` |
| 3 | Salah ketik harga ×10 | harga ≥ 7× median produknya → dibagi 10 |
| 4 | `promised_date` kosong | diisi `po_date` + median lead time produk tersebut |
| 5 | Suhu tercatat °F (nilai > 30) | dikonversi ke °C |
| — | Harga melonjak 1,5–2×, qty borongan, keterlambatan ekstrem | **tidak dihapus**: outlier yang sah (kejadian bisnis nyata), tetap dianalisis |

In [3]:
po = src_po.copy()

# 1. rapikan vendor_id
po['vendor_id'] = po.vendor_id.str.strip().str.upper()

# 2. hapus duplikat
jumlah_awal = len(po)
po = po.drop_duplicates(subset=['po_number', 'line_no']).reset_index(drop=True)
print('Duplikat dihapus          :', jumlah_awal - len(po))

# 3. perbaiki salah ketik harga x10
median_harga = po.groupby('product_id').unit_price.transform('median')
salah_ketik = po.unit_price >= 7 * median_harga
po.loc[salah_ketik, 'unit_price'] = (po.loc[salah_ketik, 'unit_price'] / 10).round(2)
print('Harga salah ketik diperbaiki:', salah_ketik.sum())

# 4. isi promised_date yang kosong
lead = (po.promised_date - po.po_date).dt.days
median_lead = lead.groupby(po.product_id).transform('median')
kosong = po.promised_date.isna()
po.loc[kosong, 'promised_date'] = po.loc[kosong, 'po_date'] + pd.to_timedelta(median_lead[kosong], unit='D')
print('promised_date diisi        :', kosong.sum())

# hitung ulang nilai uang setelah harga diperbaiki
po['gross_amount'] = (po.quantity_ordered * po.unit_price).round(2)
po['tax_amount']   = (po.gross_amount * po.tax_rate).round(2)
po['net_amount']   = (po.gross_amount + po.tax_amount).round(2)

# 5. suhu Fahrenheit -> Celsius
gr = src_gr.copy()
fahrenheit = gr.temperature_c > 30
gr.loc[fahrenheit, 'temperature_c'] = ((gr.loc[fahrenheit, 'temperature_c'] - 32) * 5 / 9).round(1)
print('Suhu °F dikonversi        :', fahrenheit.sum())

Duplikat dihapus          : 10
Harga salah ketik diperbaiki: 8
promised_date diisi        : 60
Suhu °F dikonversi        : 20


## 3. Transform (b): dim_date
Kalender harian 2023-11-01 s.d. 2026-12-31 + satu baris khusus `99991231` = "Belum Diterima"
(dipakai `delivery_date_key` bila barang belum datang, sama seperti FIX.zip).

In [4]:
NAMA_BULAN = ['Januari', 'Februari', 'Maret', 'April', 'Mei', 'Juni', 'Juli', 'Agustus', 'September', 'Oktober', 'November', 'Desember']

tgl = pd.date_range('2023-11-01', '2026-12-31')
dim_date = pd.DataFrame({
    'date_key':     tgl.strftime('%Y%m%d').astype(int),
    'full_date':    tgl.date,
    'day_of_month': tgl.day,
    'month_number': tgl.month,
    'month_name':   [NAMA_BULAN[m - 1] for m in tgl.month],
    'quarter':      tgl.quarter,
    'year':         tgl.year,
})
belum = pd.DataFrame([{'date_key': KEY_BELUM_DITERIMA, 'full_date': TGL_MAKS.date(), 'day_of_month': 31,
                       'month_number': 12, 'month_name': 'Belum Diterima', 'quarter': 4, 'year': 9999}])
dim_date = pd.concat([dim_date, belum], ignore_index=True)
date_key = lambda s: s.dt.strftime('%Y%m%d').astype(int)
dim_date.tail(3)

,date_key,full_date,day_of_month,month_number,month_name,quarter,year
1155,20261230,2026-12-30,30,12,Desember,4,2026
1156,20261231,2026-12-31,31,12,Desember,4,2026
1157,99991231,9999-12-31,31,12,Belum Diterima,4,9999


## 4. Transform (c): dim_vendor & dim_product (SCD tipe 2)
Cara kerja fungsi `bangun_scd2`:
1. Mulai dari nilai **terkini** di tabel master.
2. "Mundurkan" semua perubahan SCD 2 (pakai `old_value` di change log) → didapat nilai **awal**.
3. Maju lagi per tanggal perubahan: setiap tanggal perubahan = satu **versi baru** dengan surrogate key baru.

| Dimensi | SCD 2 (buat versi baru) | SCD 1 (timpa, ambil nilai terbaru) |
|---|---|---|
| dim_vendor | city, region, vendor_category | vendor_name |
| dim_product | product_type | product_name |

In [5]:
def bangun_scd2(master, nama_tabel, kolom_id, kolom_scd2):
    log = src_log[(src_log.table_name == nama_tabel) & (src_log.column_name.isin(kolom_scd2))]
    hasil = []
    for _, terkini in master.iterrows():
        perubahan = log[log.record_id == terkini[kolom_id]].sort_values('changed_at')
        versi = terkini.copy()
        for _, p in perubahan[::-1].iterrows():                  # mundur ke nilai awal
            versi[p.column_name] = p.old_value
        berlaku_dari = TGL_AWAL_SCD
        for tgl_ubah, grup in perubahan.groupby('changed_at'):     # maju per tanggal perubahan
            hasil.append({**versi, 'effective_date': berlaku_dari, 'expiry_date': tgl_ubah - pd.Timedelta(days=1)})
            for _, p in grup.iterrows():
                versi[p.column_name] = p.new_value
            berlaku_dari = tgl_ubah
        hasil.append({**versi, 'effective_date': berlaku_dari, 'expiry_date': TGL_MAKS})
    df = pd.DataFrame(hasil).sort_values([kolom_id, 'effective_date']).reset_index(drop=True)
    df['is_current'] = df.expiry_date == TGL_MAKS
    return df

dim_vendor = bangun_scd2(src_vendor, 'vendor', 'vendor_id', ['city', 'province', 'vendor_category'])
dim_vendor.insert(0, 'vendor_key', range(1, len(dim_vendor) + 1))
dim_vendor = dim_vendor.rename(columns={'province': 'region'})[
    ['vendor_key', 'vendor_id', 'vendor_name', 'vendor_category', 'city', 'region', 'effective_date', 'expiry_date', 'is_current']]

dim_product = bangun_scd2(src_product, 'product', 'product_id', ['product_category'])
dim_product.insert(0, 'product_key', range(1, len(dim_product) + 1))
dim_product = dim_product.rename(columns={'product_category': 'product_type'})[
    ['product_key', 'product_id', 'product_name', 'product_type', 'allocation_line', 'unit_of_measure',
     'effective_date', 'expiry_date', 'is_current']]

print('dim_vendor :', len(src_vendor), 'vendor ->', len(dim_vendor), 'baris (versi)')
print('dim_product:', len(src_product), 'produk ->', len(dim_product), 'baris (versi)')
pd.concat([dim_vendor[dim_vendor.vendor_id.isin(['VND-005', 'VND-011', 'VND-012'])]])

dim_vendor : 20 vendor -> 23 baris (versi)
dim_product: 18 produk -> 19 baris (versi)


,vendor_key,vendor_id,vendor_name,vendor_category,city,region,effective_date,expiry_date,is_current
4,5,VND-005,CV Lembang Dairy Farm,Peternakan Sapi Perah,Bandung Barat,Jawa Barat,2023-01-01,2025-02-28,False
5,6,VND-005,CV Lembang Dairy Farm,Peternakan Sapi Perah,Subang,Jawa Barat,2025-03-01,9999-12-31,True
11,12,VND-011,CV Sumber Garam Rembang,Supplier Bahan Pangan,Rembang,Jawa Tengah,2023-01-01,2025-08-14,False
12,13,VND-011,CV Sumber Garam Rembang,Supplier Bahan Pangan,Tuban,Jawa Timur,2025-08-15,9999-12-31,True
13,14,VND-012,PT Warna Alam Food Ingredients,Supplier Bahan Pangan,Surabaya,Jawa Timur,2023-01-01,2025-05-31,False
14,15,VND-012,PT Warna Alam Food Ingredients,Supplier Kultur & Enzim,Surabaya,Jawa Timur,2025-06-01,9999-12-31,True


## 5. Transform (d): dim_contract (SCD tipe 3)
Adendum kontrak hanya menyimpan **satu nilai sebelumnya**: `prev_end_date` dan `prev_max_quota_limit`.
Baris `contract_key = 0` (`CTR-NONE`) dipakai untuk pembelian spot/tanpa kontrak, sama seperti FIX.zip.

In [6]:
adendum = src_log[src_log.table_name == 'contract'].sort_values('changed_at').groupby(['record_id', 'column_name']).old_value.last()

dim_contract = src_contract.rename(columns={'quota_volume': 'max_quota_limit'}).copy()
dim_contract['prev_end_date'] = pd.to_datetime(dim_contract.contract_id.map(adendum.xs('end_date', level='column_name')))
dim_contract['prev_max_quota_limit'] = pd.to_numeric(dim_contract.contract_id.map(adendum.xs('quota_volume', level='column_name')))
dim_contract['is_active'] = dim_contract.end_date >= TGL_EKSTRAK
dim_contract = dim_contract.sort_values('contract_id').reset_index(drop=True)
dim_contract.insert(0, 'contract_key', range(1, len(dim_contract) + 1))
spot = {'contract_key': 0, 'contract_id': 'CTR-NONE', 'contract_type': 'Non-Kontrak (Spot Purchase)',
        'price_agreement_type': 'Harga Pasar', 'sla_temperature_requirement': 'Tidak Dipersyaratkan', 'is_active': False}
dim_contract = pd.concat([pd.DataFrame([spot]), dim_contract], ignore_index=True)[
    ['contract_key', 'contract_id', 'contract_type', 'price_agreement_type', 'max_quota_limit', 'sla_temperature_requirement',
     'is_active', 'start_date', 'end_date', 'prev_end_date', 'prev_max_quota_limit']]
dim_contract[dim_contract.prev_end_date.notna()].head(3)

,contract_key,contract_id,contract_type,price_agreement_type,max_quota_limit,sla_temperature_requirement,is_active,start_date,end_date,prev_end_date,prev_max_quota_limit
39,39,CTR-2025-002,SLA-Based Quality Contract,Harga Indeks Pasar,7991800.0,Rantai Dingin 2-4 °C,True,2025-01-01,2026-12-31,2025-12-31,NaN
40,40,CTR-2025-004,SLA-Based Quality Contract,Harga Indeks Pasar,7072200.0,Rantai Dingin 2-4 °C,False,2025-01-01,2026-06-30,2025-12-31,NaN
41,41,CTR-2025-006,SLA-Based Quality Contract,Harga Indeks Pasar,6576600.0,Rantai Dingin 2-4 °C,False,2025-01-01,2026-06-30,2025-12-31,NaN


## 6. Transform (e): dim_po_profile (junk dimension)
Kombinasi po_status × urgency_level × is_contract_locked = 6 × 3 × 2 = 36 baris (sama dengan FIX.zip).

In [7]:
dim_po_profile = pd.MultiIndex.from_product([
    ['Open', 'Approved', 'Partially Received', 'Received', 'Closed', 'Cancelled'],
    ['Normal', 'Urgent', 'Critical'],
    ['Yes', 'No']], names=['po_status', 'urgency_level', 'is_contract_locked']).to_frame(index=False)
dim_po_profile.insert(0, 'po_profile_key', range(1, len(dim_po_profile) + 1))
dim_po_profile.head()

,po_profile_key,po_status,urgency_level,is_contract_locked
0,1,Open,Normal,Yes
1,2,Open,Normal,No
2,3,Open,Urgent,Yes
3,4,Open,Urgent,No
4,5,Open,Critical,Yes


## 7. Transform (f): fact_purchase_order
**Grain:** satu baris = satu produk dalam satu PO → primary key gabungan `(order_id, product_key)` sesuai ERD.

1. Ambil tanggal terima dari penerimaan gudang (belum diterima → `99991231`).
2. Cari surrogate key vendor & produk **versi yang berlaku pada tanggal PO** (inti SCD 2).
3. Cari `contract_key` dan `po_profile_key`.

In [8]:
terima = gr.groupby(['po_number', 'line_no']).receive_date.max().reset_index()
f = po.merge(terima, on=['po_number', 'line_no'], how='left')

def cari_key(fakta, dim, kolom_id, kolom_key):
    """Pilih versi dimensi yang effective_date <= po_date <= expiry_date."""
    m = fakta[['po_number', 'line_no', kolom_id, 'po_date']].merge(dim[[kolom_id, kolom_key, 'effective_date', 'expiry_date']], on=kolom_id)
    m = m[(m.po_date >= m.effective_date) & (m.po_date <= m.expiry_date)]
    return fakta.merge(m[['po_number', 'line_no', kolom_key]], on=['po_number', 'line_no'], how='left')

f = cari_key(f, dim_vendor, 'vendor_id', 'vendor_key')
f = cari_key(f, dim_product, 'product_id', 'product_key')
f['contract_key'] = f.contract_id.fillna('CTR-NONE').map(dim_contract.set_index('contract_id').contract_key)
f['is_contract_locked'] = np.where(f.contract_id.notna(), 'Yes', 'No')
f = f.merge(dim_po_profile, on=['po_status', 'urgency_level', 'is_contract_locked'], how='left')

f['po_date_key'] = date_key(f.po_date)
f['delivery_date_key'] = KEY_BELUM_DITERIMA
f.loc[f.receive_date.notna(), 'delivery_date_key'] = date_key(f.loc[f.receive_date.notna(), 'receive_date'])
f['order_id'] = f.po_number.map({no: i for i, no in enumerate(sorted(f.po_number.unique()), start=100001)})

fact_purchase_order = f[['order_id', 'product_key', 'vendor_key', 'contract_key', 'po_date_key', 'delivery_date_key',
                         'po_profile_key', 'po_number', 'quantity_ordered', 'unit_price', 'gross_amount', 'tax_amount',
                         'net_amount']].sort_values(['order_id', 'product_key']).reset_index(drop=True)
print('Baris fakta           :', len(fact_purchase_order))
print('Key kosong            :', fact_purchase_order.isna().sum().sum())
print('PK (order_id, product_key) unik:', not fact_purchase_order.duplicated(['order_id', 'product_key']).any())
fact_purchase_order.head()

Baris fakta           : 10000
Key kosong            : 0
PK (order_id, product_key) unik: True


,order_id,product_key,vendor_key,contract_key,po_date_key,delivery_date_key,po_profile_key,po_number,quantity_ordered,unit_price,gross_amount,tax_amount,net_amount
0,100001,2,7,12,20240101,20240102,27,PO-202401-00001,13849.8,6444.24,8.925144e+07,0.0,8.925144e+07
1,100002,5,9,16,20240101,20240112,25,PO-202401-00002,26.0,424520.00,1.103752e+07,1214127.2,1.225165e+07
2,100003,5,9,16,20240101,20240110,25,PO-202401-00003,119.0,424520.00,5.051788e+07,5556966.8,5.607485e+07
3,100004,1,8,13,20240102,20240104,25,PO-202401-00004,10752.8,7274.77,7.822415e+07,0.0,7.822415e+07
4,100004,3,8,14,20240102,20240108,25,PO-202401-00004,4475.4,36814.04,1.647576e+08,0.0,1.647576e+08


## 8. Load: simpan ke DuckDB
DDL mengikuti tipe data di ERD (VARCHAR(50), DECIMAL(18,4) untuk qty & kuota, DECIMAL(18,2) untuk uang).
Database disimpan dengan `storage_compatibility_version = v1.0.0` agar bisa dibaca driver ODBC DuckDB versi 1.x mana pun.

In [9]:
DDL = """
CREATE TABLE dim_date (
    date_key      INTEGER PRIMARY KEY,
    full_date     DATE,
    day_of_month  INTEGER,
    month_number  INTEGER,
    month_name    VARCHAR(20),
    quarter       INTEGER,
    year          INTEGER);

CREATE TABLE dim_product (
    product_key      INTEGER PRIMARY KEY,
    product_id       VARCHAR(50),
    product_name     VARCHAR(100),
    product_type     VARCHAR(50),
    allocation_line  VARCHAR(50),
    unit_of_measure  VARCHAR(20),
    effective_date   DATE,          -- SCD 2
    expiry_date      DATE,          -- SCD 2
    is_current       BOOLEAN);      -- SCD 2

CREATE TABLE dim_vendor (
    vendor_key       INTEGER PRIMARY KEY,
    vendor_id        VARCHAR(50),
    vendor_name      VARCHAR(100),
    vendor_category  VARCHAR(50),
    city             VARCHAR(50),
    region           VARCHAR(50),
    effective_date   DATE,          -- SCD 2
    expiry_date      DATE,          -- SCD 2
    is_current       BOOLEAN);      -- SCD 2

CREATE TABLE dim_contract (
    contract_key                 INTEGER PRIMARY KEY,
    contract_id                  VARCHAR(50),
    contract_type                VARCHAR(50),
    price_agreement_type         VARCHAR(50),
    max_quota_limit              DECIMAL(18,4),
    sla_temperature_requirement  VARCHAR(50),
    is_active                    BOOLEAN,
    start_date                   DATE,
    end_date                     DATE,
    prev_end_date                DATE,            -- SCD 3
    prev_max_quota_limit         DECIMAL(18,4));  -- SCD 3

CREATE TABLE dim_po_profile (
    po_profile_key      INTEGER PRIMARY KEY,
    po_status           VARCHAR(30),
    urgency_level       VARCHAR(20),
    is_contract_locked  VARCHAR(5));

CREATE TABLE fact_purchase_order (
    order_id           INTEGER,
    product_key        INTEGER REFERENCES dim_product(product_key),
    vendor_key         INTEGER REFERENCES dim_vendor(vendor_key),
    contract_key       INTEGER REFERENCES dim_contract(contract_key),
    po_date_key        INTEGER REFERENCES dim_date(date_key),
    delivery_date_key  INTEGER REFERENCES dim_date(date_key),
    po_profile_key     INTEGER REFERENCES dim_po_profile(po_profile_key),
    po_number          VARCHAR(50),
    quantity_ordered   DECIMAL(18,4),
    unit_price         DECIMAL(18,2),
    gross_amount       DECIMAL(18,2),
    tax_amount         DECIMAL(18,2),
    net_amount         DECIMAL(18,2),
    PRIMARY KEY (order_id, product_key));
"""

TABEL = {'dim_date': dim_date, 'dim_product': dim_product, 'dim_vendor': dim_vendor, 'dim_contract': dim_contract,
         'dim_po_profile': dim_po_profile, 'fact_purchase_order': fact_purchase_order}   # dimensi dulu, fakta terakhir

if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
with duckdb.connect(DB_PATH, config={'storage_compatibility_version': 'v1.0.0'}) as con:
    con.execute(DDL)
    for nama, df in TABEL.items():
        con.register('tmp', df)
        con.execute(f'INSERT INTO {nama} SELECT {", ".join(df.columns)} FROM tmp')
        con.unregister('tmp')
    con.execute('CHECKPOINT')
    print(con.sql("SELECT table_name, estimated_size AS perkiraan_baris FROM duckdb_tables() ORDER BY table_name"))
# koneksi ditutup otomatis (with) -> file tidak terkunci saat dibuka Power BI

os.makedirs(CSV_DIR, exist_ok=True)       # cadangan CSV untuk Power BI tanpa driver
for nama, df in TABEL.items():
    df.to_csv(f'{CSV_DIR}/{nama}.csv', index=False)

┌─────────────────────┬─────────────────┐
│     table_name      │ perkiraan_baris │
│       varchar       │      int64      │
├─────────────────────┼─────────────────┤
│ dim_contract        │              77 │
│ dim_date            │            1158 │
│ dim_po_profile      │              36 │
│ dim_product         │              19 │
│ dim_vendor          │              23 │
│ fact_purchase_order │           10000 │
└─────────────────────┴─────────────────┘



## 9. Validasi data mart

In [10]:
with duckdb.connect(DB_PATH, read_only=True) as con:
    print(con.sql("""
        SELECT 'fact_purchase_order' AS tabel, COUNT(*) AS baris FROM fact_purchase_order
        UNION ALL SELECT 'dim_vendor', COUNT(*) FROM dim_vendor
        UNION ALL SELECT 'dim_product', COUNT(*) FROM dim_product
        UNION ALL SELECT 'dim_contract', COUNT(*) FROM dim_contract
        UNION ALL SELECT 'dim_po_profile', COUNT(*) FROM dim_po_profile
        UNION ALL SELECT 'dim_date', COUNT(*) FROM dim_date"""))
    # bukti SCD 2: transaksi VND-005 sebelum & sesudah pindah ke Subang tercatat di kota yang berbeda
    print(con.sql("""
        SELECT v.vendor_id, v.city, v.effective_date, v.expiry_date, COUNT(*) AS jumlah_baris_po
        FROM fact_purchase_order f JOIN dim_vendor v USING (vendor_key)
        WHERE v.vendor_id = 'VND-005' GROUP BY ALL ORDER BY v.effective_date"""))

┌─────────────────────┬───────┐
│        tabel        │ baris │
│       varchar       │ int64 │
├─────────────────────┼───────┤
│ fact_purchase_order │ 10000 │
│ dim_vendor          │    23 │
│ dim_product         │    19 │
│ dim_contract        │    77 │
│ dim_po_profile      │    36 │
│ dim_date            │  1158 │
└─────────────────────┴───────┘

┌───────────┬───────────────┬────────────────┬─────────────┬─────────────────┐
│ vendor_id │     city      │ effective_date │ expiry_date │ jumlah_baris_po │
│  varchar  │    varchar    │      date      │    date     │      int64      │
├───────────┼───────────────┼────────────────┼─────────────┼─────────────────┤
│ VND-005   │ Bandung Barat │ 2023-01-01     │ 2025-02-28  │             265 │
│ VND-005   │ Subang        │ 2025-03-01     │ 9999-12-31  │             408 │
└───────────┴───────────────┴────────────────┴─────────────┴─────────────────┘



### Uji query analitik dari FIX.zip
Query di `pertanyaan_analitik_procurement.sql` dijalankan ke data mart baru untuk memastikan skemanya kompatibel.

In [11]:
SQL_FILE = 'pertanyaan_analitik_procurement.sql'
if os.path.exists(SQL_FILE):
    blok = open(SQL_FILE, encoding='utf-8').read().split('-- ============ ')[1:]
    with duckdb.connect(DB_PATH, read_only=True) as con:
        for b in blok:
            judul, query = b.split(' ============', 1)
            print(f'--- {judul} ---')
            print(con.sql(query.strip().rstrip(';')).df().head(3).to_string(index=False), '\n')
else:
    print('Letakkan pertanyaan_analitik_procurement.sql (dari FIX.zip) di folder ini untuk menjalankan uji query.')

--- Q1 ---
 year  jumlah_po  vendor_aktif  belanja_miliar_rp
 2024       2566            20             335.60
 2025       2663            23             362.25
 2026       2031            20             286.14 

--- Q2 ---
 month_number month_name  belanja_miliar_rp
            1    Januari              34.36
            2   Februari              25.65
            3      Maret              29.42 

--- Q3 ---
                               vendor_name       vendor_category  belanja_miliar_rp  pangsa_persen
Koperasi Peternak Sapi Perah Merapi Makmur         Koperasi Susu              47.34           13.1
        KUD Sari Mulya Pangalengan Bandung         Koperasi Susu              42.06           11.6
             PT Peternakan Sapi Nusa Hijau Peternakan Sapi Perah              40.19           11.1 

--- Q4 ---
      vendor_category  jumlah_po  rata2_nilai_po_juta  nilai_po_maks_juta
Peternakan Sapi Perah       1433                184.5              2026.4
        Koperasi Susu       22

      po_number                                vendor_name tanggal_po  jumlah_item  nilai_po_juta
PO-202501-00201 Koperasi Peternak Sapi Perah Merapi Makmur 2025-01-30            1         2465.9
PO-202402-00190 Koperasi Peternak Sapi Perah Merapi Makmur 2024-02-28            2         2336.7
PO-202602-00023                      CV Lembang Dairy Farm 2026-02-05            1         2026.4 



---
# 10. Paket koneksi Power BI
Sel berikut membuat folder **`koneksi_powerbi/`** berisi:

| File | Fungsi |
|---|---|
| `1_buat_dsn_duckdb.ps1` | Membuat ODBC DSN `DuckDB_Procurement` otomatis (mode read-only) |
| `2_power_query.pq` | Kode Power Query (M) untuk memuat 6 tabel + tipe datanya |
| `3_measures.dax` | Kolom hitung & measure DAX untuk business questions |

In [12]:
KIT = 'koneksi_powerbi'
os.makedirs(KIT, exist_ok=True)

# ---------- 1. skrip PowerShell pembuat DSN ----------
ps1 = r"""# ============================================================
#  Membuat ODBC DSN "DuckDB_Procurement" untuk Power BI
#  Syarat : driver ODBC DuckDB sudah terpasang (odbc_install.exe)
#  Cara   : klik kanan file ini > Run with PowerShell
#           atau: powershell -ExecutionPolicy Bypass -File 1_buat_dsn_duckdb.ps1
# ============================================================
$ErrorActionPreference = 'Stop'
$DsnName = 'DuckDB_Procurement'
$Driver  = 'DuckDB Driver'

# file .duckdb dicari di folder induk (satu tingkat di atas folder koneksi_powerbi)
$DbPath = Join-Path (Split-Path $PSScriptRoot -Parent) 'procurement_datamart.duckdb'
if (-not (Test-Path $DbPath)) {
    $DbPath = Read-Host 'File tidak ditemukan. Masukkan path lengkap procurement_datamart.duckdb'
}
$DbPath = (Resolve-Path $DbPath).Path

if (-not (Get-OdbcDriver -Name $Driver -Platform '64-bit' -ErrorAction SilentlyContinue)) {
    Write-Host "Driver '$Driver' belum terpasang." -ForegroundColor Red
    Write-Host 'Unduh dari https://github.com/duckdb/duckdb-odbc/releases (windows-amd64), ekstrak, jalankan odbc_install.exe sebagai administrator.'
    Read-Host 'Tekan Enter untuk keluar'; exit 1
}

# hapus DSN lama (jika ada), lalu buat ulang
Get-OdbcDsn -Name $DsnName -DsnType User -Platform '64-bit' -ErrorAction SilentlyContinue | Remove-OdbcDsn
Add-OdbcDsn -Name $DsnName -DriverName $Driver -DsnType User -Platform '64-bit' `
            -SetPropertyValue @("Database=$DbPath", 'access_mode=READ_ONLY')

Write-Host ''
Write-Host "DSN '$DsnName' berhasil dibuat" -ForegroundColor Green
Write-Host "Database : $DbPath"
Write-Host 'Di Power BI: Get Data > ODBC > pilih DSN DuckDB_Procurement'
Read-Host 'Tekan Enter untuk keluar'
"""
open(f'{KIT}/1_buat_dsn_duckdb.ps1', 'w', encoding='utf-8-sig').write(ps1)

# ---------- 2. Power Query (M) ----------
TIPE_M = {'INTEGER': 'Int64.Type', 'DATE': 'type date', 'BOOLEAN': 'type logical', 'VARCHAR': 'type text',
          'DECIMAL(18,4)': 'type number', 'DECIMAL(18,2)': 'Currency.Type'}
def tipe_kolom(nama_tabel):
    with duckdb.connect(DB_PATH, read_only=True) as con:
        kol = con.sql(f"SELECT column_name, data_type FROM information_schema.columns WHERE table_name = '{nama_tabel}' ORDER BY ordinal_position").fetchall()
    return ', '.join('{"%s", %s}' % (c, TIPE_M.get(t, TIPE_M['VARCHAR'] if t.startswith('VARCHAR') else 'type any')) for c, t in kol)

pq = ['// ================================================================',
      '// Power Query untuk Data Mart Procurement (DuckDB via ODBC)',
      '// Cara pakai: Home > Transform data > New Source > Blank Query >',
      '//   Advanced Editor > tempel SATU blok per query > ganti nama query sesuai judul blok.',
      '// Buat dulu query parameter "KoneksiDuckDB".',
      '// ================================================================', '',
      '// ===== Query: KoneksiDuckDB  (parameter) =====',
      '// Pilihan 1 (pakai DSN dari skrip 1_buat_dsn_duckdb.ps1):',
      '"dsn=DuckDB_Procurement" meta [IsParameterQuery = true, Type = "Text", IsParameterQueryRequired = true]',
      '// Pilihan 2 (tanpa DSN, ganti path sesuai lokasi file):',
      '// "driver={DuckDB Driver};database=C:\\DWBI\\procurement_datamart.duckdb;access_mode=READ_ONLY" meta [IsParameterQuery = true, Type = "Text", IsParameterQueryRequired = true]', '']
for t in TABEL:
    pq += [f'// ===== Query: {t} =====', 'let',
           f'    Sumber = Odbc.Query(KoneksiDuckDB, "SELECT * FROM {t}"),',
           f'    TipeData = Table.TransformColumnTypes(Sumber, {{{tipe_kolom(t)}}})',
           'in', '    TipeData', '']
open(f'{KIT}/2_power_query.pq', 'w', encoding='utf-8').write('\n'.join(pq))

# ---------- 3. DAX ----------
dax = r"""// ================================================================
// KOLOM HITUNG (Modeling > New column) di tabel fact_purchase_order
// ================================================================
Tanggal PO =
    DATE ( INT ( fact_purchase_order[po_date_key] / 10000 ),
           MOD ( INT ( fact_purchase_order[po_date_key] / 100 ), 100 ),
           MOD ( fact_purchase_order[po_date_key], 100 ) )

Tanggal Terima =
    IF ( fact_purchase_order[delivery_date_key] <> 99991231,
         DATE ( INT ( fact_purchase_order[delivery_date_key] / 10000 ),
                MOD ( INT ( fact_purchase_order[delivery_date_key] / 100 ), 100 ),
                MOD ( fact_purchase_order[delivery_date_key], 100 ) ) )

Lead Time (hari) =
    IF ( NOT ISBLANK ( fact_purchase_order[Tanggal Terima] ),
         DATEDIFF ( fact_purchase_order[Tanggal PO], fact_purchase_order[Tanggal Terima], DAY ) )

// ================================================================
// MEASURE (Modeling > New measure), simpan di tabel _Measures
// ================================================================
Total Belanja =
    CALCULATE ( SUM ( fact_purchase_order[net_amount] ), dim_po_profile[po_status] <> "Cancelled" )

Total PPN =
    CALCULATE ( SUM ( fact_purchase_order[tax_amount] ), dim_po_profile[po_status] <> "Cancelled" )

Jumlah PO =
    CALCULATE ( DISTINCTCOUNT ( fact_purchase_order[order_id] ), dim_po_profile[po_status] <> "Cancelled" )

Volume Dipesan =
    CALCULATE ( SUM ( fact_purchase_order[quantity_ordered] ), dim_po_profile[po_status] <> "Cancelled" )

Harga Rata-rata Tertimbang =
    DIVIDE ( SUM ( fact_purchase_order[gross_amount] ), SUM ( fact_purchase_order[quantity_ordered] ) )

Rata-rata Lead Time (hari) =
    AVERAGE ( fact_purchase_order[Lead Time (hari)] )

Belanja per Tanggal Terima =        // memakai relasi TIDAK aktif delivery_date_key -> dim_date
    CALCULATE ( [Total Belanja],
                USERELATIONSHIP ( fact_purchase_order[delivery_date_key], dim_date[date_key] ) )

Porsi Belanja Spot % =
    DIVIDE ( CALCULATE ( [Total Belanja], dim_po_profile[is_contract_locked] = "No" ), [Total Belanja] )

Kepatuhan Kontrak % =
    DIVIDE ( CALCULATE ( [Jumlah PO], dim_po_profile[is_contract_locked] = "Yes" ), [Jumlah PO] )

Pemenuhan Kuota % =                 // pakai di tabel/visual per dim_contract[contract_id]
    DIVIDE ( [Volume Dipesan], SUM ( dim_contract[max_quota_limit] ) )

Kontrak Berakhir 30-60 Hari =
    VAR Hari_Ini = DATE ( 2026, 9, 25 )     // tanggal ekstraksi data; ganti TODAY() bila data live
    RETURN CALCULATE ( COUNTROWS ( dim_contract ),
                       dim_contract[end_date] >= Hari_Ini + 30,
                       dim_contract[end_date] <= Hari_Ini + 60 )

Pertumbuhan Belanja YoY % =
    VAR Tahun_Ini  = [Total Belanja]
    VAR Tahun_Lalu = CALCULATE ( [Total Belanja], SAMEPERIODLASTYEAR ( dim_date[full_date] ) )
    RETURN DIVIDE ( Tahun_Ini - Tahun_Lalu, Tahun_Lalu )
"""
open(f'{KIT}/3_measures.dax', 'w', encoding='utf-8').write(dax)
print(sorted(os.listdir(KIT)))
print(open(f'{KIT}/2_power_query.pq', encoding='utf-8').read()[-900:])

['1_buat_dsn_duckdb.ps1', '2_power_query.pq', '3_measures.dax']
peData

// ===== Query: dim_po_profile =====
let
    Sumber = Odbc.Query(KoneksiDuckDB, "SELECT * FROM dim_po_profile"),
    TipeData = Table.TransformColumnTypes(Sumber, {{"po_profile_key", Int64.Type}, {"po_status", type text}, {"urgency_level", type text}, {"is_contract_locked", type text}})
in
    TipeData

// ===== Query: fact_purchase_order =====
let
    Sumber = Odbc.Query(KoneksiDuckDB, "SELECT * FROM fact_purchase_order"),
    TipeData = Table.TransformColumnTypes(Sumber, {{"order_id", Int64.Type}, {"product_key", Int64.Type}, {"vendor_key", Int64.Type}, {"contract_key", Int64.Type}, {"po_date_key", Int64.Type}, {"delivery_date_key", Int64.Type}, {"po_profile_key", Int64.Type}, {"po_number", type text}, {"quantity_ordered", type number}, {"unit_price", Currency.Type}, {"gross_amount", Currency.Type}, {"tax_amount", Currency.Type}, {"net_amount", Currency.Type}})
in
    TipeData



---
# 11. Langkah koneksi ke Power BI Desktop

Siapkan satu folder tetap, misalnya `C:\DWBI\`, berisi `procurement_datamart.duckdb` dan folder `koneksi_powerbi\`.
**Tutup notebook/Jupyter** yang masih membuka file `.duckdb` sebelum membuka Power BI (DuckDB mengunci file yang sedang dipakai).

### Langkah 1 — Pasang driver ODBC DuckDB (sekali per laptop)
1. Unduh `duckdb_odbc-windows-amd64.zip` dari https://github.com/duckdb/duckdb-odbc/releases.
2. Ekstrak ke folder permanen, misalnya `C:\duckdb_odbc\` (jangan dihapus setelah instalasi).
3. Klik kanan `odbc_install.exe` → **Run as administrator**.

### Langkah 2 — Buat DSN
* **Otomatis:** klik kanan `koneksi_powerbi\1_buat_dsn_duckdb.ps1` → **Run with PowerShell**.
* **Manual:** buka **ODBC Data Sources (64-bit)** → tab *User DSN* → **Add** → pilih **DuckDB Driver** →
  Name `DuckDB_Procurement`, Database `C:\DWBI\procurement_datamart.duckdb`.

### Langkah 3 — Ambil data di Power BI
1. **Home → Get Data → More… → Other → ODBC → Connect**.
2. *Data source name (DSN)*: **DuckDB_Procurement** → OK. Bila diminta kredensial pilih **Default or Custom** → Connect.
3. Di Navigator centang 6 tabel: `fact_purchase_order`, `dim_product`, `dim_vendor`, `dim_contract`, `dim_po_profile`, `dim_date` → **Load**.

*Alternatif tanpa DSN:* di jendela ODBC pilih DSN **(None)** → *Advanced options* → Connection string:
`driver={DuckDB Driver};database=C:\DWBI\procurement_datamart.duckdb;access_mode=READ_ONLY`.
Atau pakai `2_power_query.pq` (Blank Query → Advanced Editor) agar tipe data langsung benar.

### Langkah 4 — Atur relasi (Model view) sesuai ERD
| Dari `fact_purchase_order` | Ke | Status relasi |
|---|---|---|
| `product_key` | `dim_product[product_key]` | aktif, many-to-one |
| `vendor_key` | `dim_vendor[vendor_key]` | aktif, many-to-one |
| `contract_key` | `dim_contract[contract_key]` | aktif, many-to-one |
| `po_profile_key` | `dim_po_profile[po_profile_key]` | aktif, many-to-one |
| `po_date_key` | `dim_date[date_key]` | **aktif** |
| `delivery_date_key` | `dim_date[date_key]` | **tidak aktif** (garis putus-putus) → dipakai lewat `USERELATIONSHIP` |

Power BI hanya mengizinkan satu relasi aktif antara dua tabel, maka `delivery_date_key` dibuat tidak aktif.
Cara lain: duplikat `dim_date` di Power Query menjadi `dim_date_delivery` lalu relasikan secara aktif.

### Langkah 5 — Rapikan model
* `dim_date`: pilih kolom `month_name` → **Sort by column** → `month_number`; tandai tabel sebagai **Mark as date table** (kolom `full_date`).
  Baris 9999-12-31 perlu disaring dulu di Power Query (`year <> 9999`) sebelum *Mark as date table*, atau lewati langkah ini.
* Sembunyikan kolom `*_key` di tabel fakta (klik kanan → Hide in report view).
* `dim_vendor[city]` → Data category **City**; `dim_vendor[region]` → **State or Province** (untuk visual peta).
* Buat tabel kosong `_Measures` (Enter data) lalu tempel measure dari `3_measures.dax`.
* Untuk laporan "kondisi vendor terkini", filter `dim_vendor[is_current] = TRUE`; untuk analisis historis biarkan tanpa filter (SCD 2).

### Langkah 6 — Refresh
Setiap notebook ETL dijalankan ulang, file `.duckdb` diperbarui → di Power BI klik **Refresh**. Tidak perlu membuat koneksi ulang.

### Cadangan tanpa driver
**Get Data → Text/CSV** dari folder `datamart_csv/` (isi & struktur sama), lalu lanjut ke Langkah 4.

### Troubleshooting
| Gejala | Solusi |
|---|---|
| *IO Error: Could not set lock on file* | File sedang dibuka proses lain (Jupyter, DBeaver). Tutup koneksinya; DSN sudah memakai `access_mode=READ_ONLY`. |
| Driver DuckDB tidak muncul | Pakai **ODBC Data Sources (64-bit)**; jalankan ulang `odbc_install.exe` sebagai administrator. |
| Skrip `.ps1` tidak mau jalan | Buka PowerShell di folder `koneksi_powerbi` lalu jalankan `powershell -ExecutionPolicy Bypass -File 1_buat_dsn_duckdb.ps1`. |
| Tabel tidak muncul di Navigator | Path `Database` di DSN salah; harus path lengkap ke file `.duckdb`. |
| Jumlah PO dobel | Grain per item → gunakan `DISTINCTCOUNT(order_id)`. |